In [1]:
from __future__ import annotations
from pathlib import Path 

import yaml, json, os, sys

import numpy as np, matplotlib.pyplot as plt
import igraph as ig, networkx as nx
import torch

from collections import deque 
from matplotlib.ticker import MultipleLocator, FuncFormatter
import matplotlib.colors as mcolors


ROOT = "/scratch/sleonard/MoE_circuits"
sys.path.insert(0, ROOT)

with open(os.path.join(ROOT, "config.yaml")) as f:
    config = yaml.safe_load(f)

/scratch/sleonard/miniconda3/envs/megatron/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
olmoe: dict    = torch.load(os.path.join(config["result_path"], "circuits/dag_olmoe_c4.pt"),            map_location='cpu')
deepseek: dict = torch.load(os.path.join(config["result_path"], "circuits/dag_deepseek-v2-lite_c4.pt"), map_location='cpu')

In [ ]:
from experiments.circuits.helper import get_thresholds, thresholding_routing_graph, show_enhanced_layered_graph

TARGET = "AARV"
Q = 0.99

t_olmoe = get_thresholds(olmoe, TARGET, [Q])[Q]
g_olmoe = thresholding_routing_graph(olmoe, TARGET, t_olmoe)
show_enhanced_layered_graph(g_olmoe, quantile=Q, target=TARGET, dataset="c4 (OLMoE)", n_prompts=olmoe["n_prompts"])

t_deepseek = get_thresholds(deepseek, TARGET, [Q])[Q]
g_deepseek = thresholding_routing_graph(deepseek, TARGET, t_deepseek)
show_enhanced_layered_graph(g_deepseek, quantile=Q, target=TARGET, dataset="c4 (DeepSeek-V2-Lite)", n_prompts=deepseek["n_prompts"])

In [ ]:
A_c4 = np.array(dag_c4.get_adjacency(attribute="weight").data, dtype=float)
A_code = np.array(dag_code.get_adjacency(attribute="weight").data, dtype=float)
A_math = np.array(dag_math.get_adjacency(attribute="weight").data, dtype=float)

In [ ]:
U_c4, S_c4, Vt_c4 = np.linalg.svd(A_c4, full_matrices=False)
U_code, S_code, Vt_code = np.linalg.svd(A_code, full_matrices=False)
U_math, S_math, Vt_math = np.linalg.svd(A_math, full_matrices=False)

In [ ]:
k = 100
x = np.arange(start=1, stop=k+1, step=1)
plt.plot(x, S_c4[:k], label="c4", marker='s')
plt.plot(x, S_code[:k], label="code", marker='o')
plt.plot(x, S_math[:k], label="math", marker='D')

plt.xlabel("Singular Value k")
plt.ylabel("Singular Value Magntidue")
plt.legend()
plt.show()

In [ ]:
# make a sweep: threshold x subspace_similarity
# i don't even know whether the subspace similarity is a good metric ... 

Uc4   = U_c4[:, :k]
Ucode = U_code[:, :k]
Umath = U_math[:, :k]

def subspace_similarity(U1, U2):
    M = U1.T @ U2
    s = np.linalg.svd(M, compute_uv=False)

    angles = np.arccos(np.clip(s, -1.0, 1.0))
    similarity = s.mean()

    return similarity, s, angles

sim_c4_code, s_c4_code, ang_c4_code = subspace_similarity(Uc4, Ucode)
sim_c4_math, s_c4_math, ang_c4_math = subspace_similarity(Uc4, Umath)
sim_code_math, s_code_math, ang_code_math = subspace_similarity(Ucode, Umath)

print("c4 ↔ code:", sim_c4_code)
print("c4 ↔ math:", sim_c4_math)
print("code ↔ math:", sim_code_math)

In [ ]:
!grep -n "N_LAYERS" experiments/circuits/helper.py